# Unit 14 · Classes II — Objects That Feel Native

**Learn with Adi — Python Programming (Intermediate · Object-oriented Python)**

Unit 13's objects worked, but `print` couldn't describe them, `==` couldn't compare them and `len()` wouldn't count them. This notebook mirrors the practice from Unit 14: the double-underscore methods — *dunders* — that Python is already looking for. Run a cell with **Shift+Enter**.

Study notes: https://aditya-402.github.io/learn-with-adi/series/python-programming/unit14.html

## 14.1 · The ugly print problem — `__str__`

Print an object nobody taught to describe itself and you get `<__main__.Money object at 0x...>`. The address is different every run — that is the whole point. `__str__` must **return** a string, never print one.

(`__repr__` is its developer-facing twin: `__str__` for humans, `__repr__` for debugging. `__str__` is enough for now.)

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

class Cash:
    def __init__(self, amount):
        self.amount = amount
    def __str__(self):
        return f"{self.amount} rupees"

print(Cash(250))     # your sentence
print(Money(250))    # Python's fallback — address varies every run

In [ ]:
# P1 — give Dog a __str__ so it prints:  Bruno the beagle
class Dog:
    def __init__(self, name, breed):
        self.name = name
        self.breed = breed
    # add __str__ here

d = Dog("Bruno", "beagle")
print(d)

## 14.2 · The equality surprise — `__eq__`

By default `==` asks **which box**, not what is inside it. Two separately-built `Point(3, 4)` objects are two boxes, so the honest default answer is `False`. Writing `__eq__` changes the question.

In [ ]:
class Plain:
    def __init__(self, x):
        self.x = x

class Taught:
    def __init__(self, x):
        self.x = x
    def __eq__(self, other):
        return self.x == other.x

print(Plain(7) == Plain(7))
print(Taught(7) == Taught(7))
print(Taught(7) == Taught(8))

In [ ]:
# P2 — add __eq__ so equal coordinates count as equal.
# target ->  True  then  False
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    # add __eq__ here

print(Point(3, 4) == Point(3, 4))
print(Point(3, 4) == Point(4, 3))

## 14.3 · The protocol reveal — how `len()` finds `__len__`

Built-ins are **polite knockers**. `len(x)` knocks with `x.__len__()`, `print` knocks with `__str__`, `==` knocks with `__eq__`, `+` knocks with `__add__`. No registration, no permission — implement the name and the feature switches on.

This is why `len()` worked on strings, lists **and** dictionaries back in Beginner: not three special cases, one protocol.

In [ ]:
class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []
    def add(self, song):
        self.songs.append(song)
    def __len__(self):
        return len(self.songs)

p = Playlist("Road trip")
p.add("Highway Star")
p.add("Kaatru Veliyidai")
print(len(p))
p.add("Sultans of Swing")
print(len(p))

In [ ]:
# P3 — this Basket has a __len__ but the code never calls len().
# Predict both lines, then work out who answers the question `if b:` asks.
class Basket:
    def __init__(self):
        self.items = []
    def add(self, item):
        self.items.append(item)
    def __len__(self):
        return len(self.items)

b = Basket()
if b:
    print("Basket has things")
else:
    print("Basket is empty")

b.add("apple")
if b:
    print("Basket has things")
else:
    print("Basket is empty")

## 14.4 · Making `+` mean something — `__add__`

`a + b` makes Python ask `a.__add__(b)`. Without it: `TypeError: unsupported operand type(s) for +`. With it, `+` means whatever you decide — **as long as it builds and returns a NEW object**. `3 + 4` doesn't turn the 3 into a 7; your `+` shouldn't vandalise its inputs either.

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount
    def __add__(self, other):
        return Money(self.amount + other.amount)

tea = Money(40)
snack = Money(35)
bill = tea + snack
print(bill.amount)
print(tea.amount, snack.amount)   # both originals untouched

In [ ]:
# P3 (stretch) — this __add__ is written the wrong way: it mutates instead of building.
# Run it, watch `a` get corrupted, then fix it.  target ->  100 / 50 / 150
class Money:
    def __init__(self, amount):
        self.amount = amount
    def __add__(self, other):
        self.amount = self.amount + other.amount
        return self

a = Money(100)
b = Money(50)
c = a + b
print(a.amount)
print(b.amount)
print(c.amount)

## 14.5 · Your build — Money, and a Playlist that prints itself

Four dunders and `Money` stops feeling bolted together. Then `Playlist`, where `__str__` returns a multi-line, numbered listing — a returned string may contain `\n`, and one `print(p)` honours every one of them.

In [ ]:
# Stage 1 — the complete Money class. Don't change the script; make it work.
# target ->  12.50 rupees / 45.50 rupees / 58.00 rupees / True / False
class Money:
    def __init__(self, amount):
        self.amount = amount
    # __str__ here  ->  f"{self.amount:.2f} rupees"
    # __eq__ here
    # __add__ here

tea = Money(12.5)
cake = Money(45.5)
bill = tea + cake
print(tea)
print(cake)
print(bill)
print(bill == Money(58))
print(tea == cake)

In [ ]:
# Stage 2 — Playlist with __len__ and a numbered __str__.
# target ->  3 / Monsoon (3 songs) / 1. Rain Song / 2. Cloudburst / 3. Petrichor
class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []
    def add(self, song):
        self.songs.append(song)
    # __len__ here
    # __str__ here — heading, then numbered lines joined with \n

p = Playlist("Monsoon")
p.add("Rain Song")
p.add("Cloudburst")
p.add("Petrichor")
print(len(p))
print(p)

## Problem bank

1. **The library catalogue** — give Unit 13's `Book` a `__str__` returning `title — by author`.
2. **Distances that add up** — a `Distance(km)` class with `__str__` (`8 km`), `__eq__` and `__add__`.
3. **The shopping cart** — a `Cart` with `add(price)`, `__len__`, and a plain `total()` method; three prices from `input()`. Note the mix: counting is a protocol, but "total" isn't a built-in, so it keeps an honest English name.
4. **Stretch — Vector** — `Vector(x, y)` with `__add__` (x's and y's separately) and `__str__` printing `(3, 7)`. Keep this class: it returns in the NumPy bridge unit, and again in the LLM series where two numbers become four thousand.

In [ ]:
# your problem-bank workspace
